# Quantization and Pruning of Compact 1D-CNNs for Edge PQD Classification
**Full experiment notebook (Kaggle), version 2.** Author: Rabbuni Merugumala

v2 changes after the pilot run: 4000 training signals per class, longer training, longer pruning/QAT fine-tuning, and a fix so magnitude-pruned models always keep their target sparsity.

Run order: **Run All**. Every cell is checkpointed: if the Kaggle session stops, run all again and finished seeds are skipped.

What this notebook does:
1. Generates a 16-class synthetic PQD dataset (random initial phase; plus an extra 49.5–50.5 Hz frequency-deviation test set) (IEEE 1159 based models, parameter ranges from Machlev et al., 2022), 50 Hz, 3.2 kHz, 640 samples.
2. Trains two compact baselines: a plain 1D-CNN and 1D-CCANet-3 (Bai et al., 2026; output layer adapted to 16 classes).
3. Builds 9 variants per baseline: FP32, PTQ-INT8, QAT-INT8, structured filter pruning 50% / 75% (physical rebuild), structured pruning + QAT, unstructured magnitude pruning 50% / 75%.
4. Tests every variant **as a TFLite model** on fixed test sets at clean, 50, 40, 30, 20 and 10 dB SNR.
5. Repeats everything for 5 seeds (42–46) and writes CSV tables and 600-dpi figures.

**Settings:** Accelerator = GPU (P100 or T4). Internet = ON (for the pip install in cell 1).

In [ ]:
# Cell 1 — install (must run before TensorFlow is imported)
import importlib.metadata as md, subprocess, sys, os
os.environ["TF_USE_LEGACY_KERAS"] = "1"          # tfmot needs Keras 2 (tf_keras)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
_tfv = md.version("tensorflow")
_mm = ".".join(_tfv.split(".")[:2])
_need = []
try:
    md.version("tf_keras")
except md.PackageNotFoundError:
    _need.append(f"tf_keras~={_mm}.0")
try:
    md.version("tensorflow-model-optimization")
except md.PackageNotFoundError:
    _need.append("tensorflow-model-optimization")
if _need:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_need])
print("TensorFlow", _tfv, "| installed:", _need or "nothing needed")

In [ ]:
# Cell 2 — imports and configuration
import json, time, gzip, glob, platform, warnings, logging
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import tensorflow as tf
import tf_keras as keras
from tf_keras import layers
import tensorflow_model_optimization as tfmot
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
tf.get_logger().setLevel(logging.ERROR)

QUICK = os.environ.get("PQD_QUICK", "0") == "1"   # tiny smoke test; keep False on Kaggle

if os.path.exists("/kaggle/working"):
    ROOT = "/kaggle/working/PQD_paper"
else:
    ROOT = os.environ.get("PQD_ROOT", "PQD_paper")
for sub in ["data", "models", "results", "figures"]:
    os.makedirs(os.path.join(ROOT, sub), exist_ok=True)

# ---- signal settings (fixed by the study design) ----
F0, FS, N_CYC = 50.0, 3200.0, 10
N = int(FS / F0 * N_CYC)              # 640 samples
T = 1.0 / F0                          # one cycle = 0.02 s
t = np.arange(N) / FS
W = 2 * np.pi * F0

CLASSES = ["Normal", "Sag", "Swell", "Interruption", "Harmonics", "Flicker",
           "Osc. transient", "Imp. transient", "Notch", "Spike",
           "Sag+Harm", "Swell+Harm", "Interr.+Harm", "Flicker+Harm",
           "Flicker+Sag", "Flicker+Swell"]
NC = len(CLASSES)

# ---- dataset size and noise ----
N_TRAIN, N_VAL, N_TEST = 4000, 500, 300      # per class (test: per class per SNR level); Machlev used 4800, Bai 5000
TRAIN_SNR = (20.0, 50.0)                      # uniform, as in Machlev et al. (2022)
CLEAN_FRAC = 0.10                             # share of noise-free training signals
TEST_SNRS = [None, 50, 40, 30, 20, 10]        # None = clean
FDEV_RANGE, FDEV_SNR = (49.5, 50.5), 30       # extra out-of-distribution test: grid frequency deviation
DATA_SEEDS = {"train": 2026, "val": 2027, "test": 2028}

# ---- training ----
SEEDS = [42, 43, 44, 45, 46]
EPOCHS, PATIENCE, BATCH, LR = 120, 12, 64, 1e-3
FT_EPOCHS, FT_LR = 30, 3e-4                   # fine-tuning after each pruning step (early stopping)
QAT_EPOCHS, QAT_LR = 20, 1e-4
UNSTR_EPOCHS = 30                             # sparsity ramps over the first 60%, then fixed
PRUNE_STEPS = [0.25, 0.50, 0.75]              # iterative structured pruning schedule
UNSTR_TARGETS = [0.50, 0.75]
CALIB_N = 300                                 # PTQ calibration signals (noisy mix from train set)
LAT_RUNS, LAT_WARMUP = 300, 30

if QUICK:
    N_TRAIN, N_VAL, N_TEST = 40, 10, 10
    SEEDS = [42]
    EPOCHS, FT_EPOCHS, QAT_EPOCHS, PATIENCE, UNSTR_EPOCHS = 2, 1, 1, 2, 3
    LAT_RUNS, LAT_WARMUP, CALIB_N = 20, 5, 50

ARCHS = ["cnn", "ccanet"]
VARIANTS = ["fp32", "ptq_int8", "qat_int8", "sp50", "sp75",
            "sp50_qat", "sp75_qat", "up50", "up75"]

CONFIG = {k: v for k, v in dict(globals()).items()
          if k.isupper() and isinstance(v, (int, float, str, list, dict, tuple, type(None)))}
with open(os.path.join(ROOT, "results", "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)
gpus = tf.config.list_physical_devices("GPU")
print("ROOT:", ROOT, "| GPU:", gpus, "| QUICK:", QUICK)
print("TF", tf.__version__, "| tf_keras", keras.__version__, "| tfmot", tfmot.__version__)

In [ ]:
# Cell 3 — PQD signal generator (16 classes, parameter ranges of Machlev et al. 2022, Table I)
def u(a, b):
    """Rectangular window u(t-a) - u(t-b)."""
    return ((t >= a) & (t < b)).astype(np.float64)

def window(rng, dmin, dmax):
    d = rng.uniform(dmin, dmax)
    t1 = rng.uniform(0.0, t[-1] - d)
    return t1, t1 + d

# Every signal gets a random initial phase (the recording window is not synchronised to a zero
# crossing), and optionally a fundamental frequency other than 50 Hz (used only for the
# frequency-deviation test set). TH is the instantaneous phase angle of the fundamental.
TH = None
F_SIG = F0

def harmonic_wave(rng):
    a3, a5, a7 = rng.uniform(0.05, 0.15, 3)
    a1 = np.sqrt(1.0 - (a3**2 + a5**2 + a7**2))
    return a1*np.sin(TH) + a3*np.sin(3*TH) + a5*np.sin(5*TH) + a7*np.sin(7*TH)

def flicker_env(rng):
    af, beta = rng.uniform(0.1, 0.2), rng.uniform(5.0, 20.0)
    return 1.0 + af*np.sin(2*np.pi*beta*t)

def notch_train(rng):
    """Rectangular pulses locked to the same point of every cycle, signed by the sine (Machlev et al.)."""
    Tc = 1.0 / F_SIG
    k = rng.uniform(0.1, 0.4)
    width = rng.uniform(0.01*Tc, 0.05*Tc)
    t1 = rng.uniform(0.0, 0.5*Tc - width)
    tc = np.mod(TH, 2*np.pi) / (2*np.pi) * Tc          # time since the start of the current cycle
    p = ((tc >= t1) & (tc < t1 + width)).astype(np.float64)
    return k * np.sign(np.sin(TH)) * p

def make_signal(c, rng, f_range=None):
    global TH, F_SIG
    F_SIG = F0 if f_range is None else rng.uniform(*f_range)
    TH = 2*np.pi*F_SIG*t + rng.uniform(0.0, 2*np.pi)
    s = np.sin(TH)
    if c == 0:                                   # Normal
        a = rng.uniform(0.0, 0.04); t1, t2 = window(rng, T, 9*T)
        return (1 + rng.choice([-1, 1])*a*u(t1, t2)) * s
    if c == 1:                                   # Sag
        a = rng.uniform(0.1, 0.9); t1, t2 = window(rng, T, 9*T)
        return (1 - a*u(t1, t2)) * s
    if c == 2:                                   # Swell
        a = rng.uniform(0.1, 0.8); t1, t2 = window(rng, T, 9*T)
        return (1 + a*u(t1, t2)) * s
    if c == 3:                                   # Interruption
        a = rng.uniform(0.9, 1.0); t1, t2 = window(rng, T, 9*T)
        return (1 - a*u(t1, t2)) * s
    if c == 4:                                   # Harmonics
        return harmonic_wave(rng)
    if c == 5:                                   # Flicker
        return flicker_env(rng) * s
    if c == 6:                                   # Oscillatory transient
        a = rng.uniform(0.1, 0.8); t1, t2 = window(rng, 0.5*T, 3*T)
        tau, fn = rng.uniform(0.008, 0.040), rng.uniform(300.0, 900.0)
        return s + a*np.exp(-(t - t1)/tau)*np.sin(2*np.pi*fn*(t - t1))*u(t1, t2)
    if c == 7:                                   # Impulsive transient
        a = rng.uniform(0.1, 0.414); t1, t2 = window(rng, T/20, T/10)
        return (1 - a*u(t1, t2)) * s
    if c == 8:                                   # Notch
        return s - notch_train(rng)
    if c == 9:                                   # Spike
        return s + notch_train(rng)
    if c == 10:                                  # Sag + harmonics
        a = rng.uniform(0.1, 0.9); t1, t2 = window(rng, T, 9*T)
        return (1 - a*u(t1, t2)) * harmonic_wave(rng)
    if c == 11:                                  # Swell + harmonics
        a = rng.uniform(0.1, 0.8); t1, t2 = window(rng, T, 9*T)
        return (1 + a*u(t1, t2)) * harmonic_wave(rng)
    if c == 12:                                  # Interruption + harmonics
        a = rng.uniform(0.9, 1.0); t1, t2 = window(rng, T, 9*T)
        return (1 - a*u(t1, t2)) * harmonic_wave(rng)
    if c == 13:                                  # Flicker + harmonics
        return flicker_env(rng) * harmonic_wave(rng)
    if c == 14:                                  # Flicker + sag
        a = rng.uniform(0.1, 0.9); t1, t2 = window(rng, T, 9*T)
        return flicker_env(rng) * (1 - a*u(t1, t2)) * s
    if c == 15:                                  # Flicker + swell
        a = rng.uniform(0.1, 0.8); t1, t2 = window(rng, T, 9*T)
        return flicker_env(rng) * (1 + a*u(t1, t2)) * s
    raise ValueError(c)

def add_awgn(x, snr_db, rng):
    p = np.mean(x**2)
    return x + rng.normal(0.0, np.sqrt(p / 10**(snr_db/10)), x.shape)

def build_set(n_per_class, seed, snr=None, snr_range=None, clean_frac=0.0, f_range=None):
    rng = np.random.default_rng(seed)
    X = np.zeros((n_per_class*NC, N), np.float32)
    y = np.repeat(np.arange(NC), n_per_class).astype(np.int64)
    snrs = np.full(len(y), np.nan, np.float32)
    for i, c in enumerate(y):
        x = make_signal(int(c), rng, f_range)
        if snr is not None:
            x = add_awgn(x, snr, rng); snrs[i] = snr
        elif snr_range is not None and rng.random() >= clean_frac:
            s_db = rng.uniform(*snr_range); x = add_awgn(x, s_db, rng); snrs[i] = s_db
        X[i] = x
    p = rng.permutation(len(y))
    return X[p, :, None], y[p], snrs[p]

def snr_tag(s):
    return "clean" if s is None else f"{s}dB"

DATA = os.path.join(ROOT, "data")
if not os.path.exists(os.path.join(DATA, "test_fdev.npz")) or QUICK:
    t0 = time.time()
    Xtr, ytr, str_ = build_set(N_TRAIN, DATA_SEEDS["train"], snr_range=TRAIN_SNR, clean_frac=CLEAN_FRAC)
    Xva, yva, sva = build_set(N_VAL, DATA_SEEDS["val"], snr_range=TRAIN_SNR, clean_frac=CLEAN_FRAC)
    np.savez_compressed(os.path.join(DATA, "train.npz"), X=Xtr, y=ytr, snr=str_)
    np.savez_compressed(os.path.join(DATA, "val.npz"), X=Xva, y=yva, snr=sva)
    for j, s in enumerate(TEST_SNRS):        # each SNR level gets its own independent signals
        Xte, yte, _ = build_set(N_TEST, DATA_SEEDS["test"] + 10*j, snr=s)
        np.savez_compressed(os.path.join(DATA, f"test_{snr_tag(s)}.npz"), X=Xte, y=yte)
    Xf, yf, _ = build_set(N_TEST, DATA_SEEDS["test"] + 99, snr=FDEV_SNR, f_range=FDEV_RANGE)
    np.savez_compressed(os.path.join(DATA, "test_fdev.npz"), X=Xf, y=yf)
    print(f"Dataset generated in {time.time()-t0:.1f} s")

d = np.load(os.path.join(DATA, "train.npz")); Xtr, ytr = d["X"], d["y"]
d = np.load(os.path.join(DATA, "val.npz"));   Xva, yva = d["X"], d["y"]
TEST = {}
for s in TEST_SNRS:
    d = np.load(os.path.join(DATA, f"test_{snr_tag(s)}.npz")); TEST[snr_tag(s)] = (d["X"], d["y"])
d = np.load(os.path.join(DATA, "test_fdev.npz")); TEST["fdev"] = (d["X"], d["y"])   # 49.5-50.5 Hz, 30 dB
print("train", Xtr.shape, "val", Xva.shape, "test per SNR", TEST["clean"][0].shape)
print("class counts (train):", np.bincount(ytr))

In [ ]:
# Cell 4 — figure style and waveform figure (Fig. 2 of the paper)
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
                     "font.size": 8, "axes.linewidth": 0.6, "lines.linewidth": 0.9,
                     "savefig.dpi": 600, "savefig.bbox": "tight"})
FIG = os.path.join(ROOT, "figures")

rng_fig = np.random.default_rng(7)
fig, axs = plt.subplots(4, 4, figsize=(7.0, 5.6), sharex=True, sharey=True)
for c, ax in enumerate(axs.flat):
    ax.plot(t*1000, make_signal(c, rng_fig), color="black")
    ax.set_title(f"C{c}: {CLASSES[c]}", fontsize=7.5)
    ax.set_ylim(-2.0, 2.0); ax.grid(alpha=0.25, linewidth=0.4)
for ax in axs[-1]: ax.set_xlabel("Time (ms)")
for ax in axs[:, 0]: ax.set_ylabel("Voltage (p.u.)")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "Fig_waveforms_16class.png")); fig.savefig(os.path.join(FIG, "Fig_waveforms_16class.pdf"))
plt.close(fig)
print("saved waveform figure")

In [ ]:
# Cell 5 — baseline models
# Conv1D layers are written as mathematically identical Conv2D layers with 1 x k kernels on a
# (1, 640, 1) input. TFLite executes Conv1D this way internally, and tfmot QAT supports only the
# 2-D layers (it folds Conv2D+BatchNorm during QAT; Conv1D/MaxPooling1D are not in its registry).
def conv(w, k, name, padding="valid"):
    return layers.Conv2D(int(w), (1, k), padding=padding, use_bias=False, name=name)

def build_cnn(widths=(16, 32, 64), n_classes=NC):
    """Plain compact 1D-CNN: 3 x [Conv-BN-ReLU], wide first kernel, GAP head."""
    inp = keras.Input((N, 1), name="input")
    x = layers.Reshape((1, N, 1), name="to2d")(inp)
    for i, (w, k) in enumerate(zip(widths, (7, 5, 3))):
        x = conv(w, k, f"conv{i+1}", padding="same")(x)
        x = layers.BatchNormalization(name=f"bn{i+1}")(x)
        x = layers.ReLU(name=f"relu{i+1}")(x)
        if i < 2:
            x = layers.MaxPooling2D((1, 4), name=f"pool{i+1}")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="drop")(x)
    out = layers.Dense(n_classes, activation="softmax", name="fc")(x)
    return keras.Model(inp, out, name="cnn")

def build_ccanet(widths=(6, 16, 16), n_classes=NC):
    """1D-CCANet-3 of Bai et al. (2026), Table 4: 3 conv layers (k=5, no bias),
    ECA-style cross-channel attention (k=3) + BN, flatten, bias-free FC."""
    inp = keras.Input((N, 1), name="input")
    x = layers.Reshape((1, N, 1), name="to2d")(inp)
    x = conv(widths[0], 5, "conv1")(x)
    x = layers.ReLU(name="relu1")(x)
    x = layers.MaxPooling2D((1, 3), (1, 3), name="pool1")(x)
    x = conv(widths[1], 5, "conv2")(x)
    x = layers.ReLU(name="relu2")(x)
    x = layers.MaxPooling2D((1, 3), (1, 3), name="pool2")(x)
    f = conv(widths[2], 5, "conv3")(x)                                           # (1, L, C)
    C = int(widths[2])
    a = layers.GlobalAveragePooling2D(name="eca_gap")(f)                         # (C,)
    a = layers.Reshape((1, C, 1), name="eca_r1")(a)                              # channels as a sequence
    a = layers.Conv2D(1, (1, 3), padding="same", use_bias=False, name="eca_conv")(a)
    a = layers.Activation("sigmoid", name="eca_sig")(a)
    a = layers.Reshape((1, 1, C), name="eca_r2")(a)
    x = layers.Multiply(name="eca_mul")([f, a])
    x = layers.BatchNormalization(name="bn")(x)
    x = layers.ReLU(name="relu3")(x)
    x = layers.Flatten(name="flat")(x)
    out = layers.Dense(n_classes, use_bias=False, activation="softmax", name="fc")(x)
    return keras.Model(inp, out, name="ccanet")

BUILDERS = {"cnn": build_cnn, "ccanet": build_ccanet}
BASE_WIDTHS = {"cnn": (16, 32, 64), "ccanet": (6, 16, 16)}

def n_trainable(m):
    return int(sum(np.prod(v.shape) for v in m.trainable_weights))

for a in ARCHS:
    m = BUILDERS[a]()
    print(f"{a}: trainable params = {n_trainable(m):,}")
# check: CCANet with 10 classes must give Bai's 12,225 parameters
print("CCANet-3 (10 classes) trainable params:", n_trainable(build_ccanet(n_classes=10)), "(Bai et al.: 12,225)")

In [ ]:
# Cell 6 — training helpers
def compile_model(m, lr):
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

def fit(m, epochs, lr, extra_cb=(), patience=PATIENCE, early_stop=True):
    compile_model(m, lr)
    cbs = [keras.callbacks.ReduceLROnPlateau("val_loss", factor=0.5, patience=max(2, patience//2), min_lr=1e-6),
           *extra_cb]
    if early_stop:   # never used for magnitude pruning: restoring a "best" epoch could undo the sparsity
        cbs.insert(0, keras.callbacks.EarlyStopping("val_loss", patience=patience, restore_best_weights=True))
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=epochs, batch_size=BATCH,
              shuffle=True, verbose=0, callbacks=cbs)
    return h

In [ ]:
# Cell 7 — structured filter pruning with physical rebuild (L1-norm ranking, Li et al. 2017)
CONV_NAMES = ["conv1", "conv2", "conv3"]

def structured_prune(model, arch, frac):
    """Remove the lowest-L1 filters of every conv layer so that each keeps
    round((1-frac) * original width) filters, rebuild a smaller dense model, copy weights."""
    base = BASE_WIDTHS[arch]
    keep = [max(2, int(round(w*(1-frac)))) for w in base]
    idx = []
    for name, k in zip(CONV_NAMES, keep):
        K = model.get_layer(name).get_weights()[0]                 # (1, k, in, out)
        l1 = np.abs(K).sum(axis=(0, 1, 2))
        idx.append(np.sort(np.argsort(l1)[::-1][:k]))
    new = BUILDERS[arch](widths=keep)
    prev = np.arange(1)                                            # input has 1 channel
    for name, ix in zip(CONV_NAMES, idx):
        K = model.get_layer(name).get_weights()[0]
        new.get_layer(name).set_weights([K[:, :, prev, :][:, :, :, ix]])
        prev = ix
    if arch == "cnn":
        for i, ix in enumerate(idx):
            new.get_layer(f"bn{i+1}").set_weights([w[ix] for w in model.get_layer(f"bn{i+1}").get_weights()])
        Wd, bd = model.get_layer("fc").get_weights()
        new.get_layer("fc").set_weights([Wd[idx[2], :], bd])
    else:
        new.get_layer("eca_conv").set_weights(model.get_layer("eca_conv").get_weights())
        new.get_layer("bn").set_weights([w[idx[2]] for w in model.get_layer("bn").get_weights()])
        Wd = model.get_layer("fc").get_weights()[0]                 # (L*C, n), flatten order = (time, channel)
        L = model.get_layer("conv3").output_shape[2]
        C = base[2] if Wd.shape[0] == L*base[2] else Wd.shape[0] // L
        new.get_layer("fc").set_weights([Wd.reshape(L, C, -1)[:, idx[2], :].reshape(L*len(idx[2]), -1)])
    return new, keep

In [ ]:
# Cell 8 — QAT and unstructured pruning wrappers (tfmot)
from tensorflow_model_optimization.python.core.quantization.keras.default_8bit import (
    default_8bit_quantize_registry, default_8bit_quantize_configs)
_OUTCFG = default_8bit_quantize_configs.Default8BitOutputQuantizeConfig
_REG = default_8bit_quantize_registry.Default8BitQuantizeRegistry()

def fresh_copy(model):
    """Independent copy (new layer objects), so later steps never modify the source model."""
    c = keras.models.clone_model(model)
    c.set_weights(model.get_weights())
    return c

def make_qat(model):
    """Whole-model QAT (with Conv+BN folding) when every layer is supported by tfmot.
    Otherwise (Multiply and the stand-alone BN of the CCANet attention block, which tfmot does
    not support), those layers get an output-only fake-quantizer, so QAT still models every
    INT8 tensor that the converted TFLite model will use."""
    model = fresh_copy(model)
    try:
        return tfmot.quantization.keras.quantize_model(model), "full"
    except Exception:
        def annot(layer):
            if _REG.supports(layer):
                return tfmot.quantization.keras.quantize_annotate_layer(layer)
            if isinstance(layer, (layers.Multiply, layers.BatchNormalization)):
                return tfmot.quantization.keras.quantize_annotate_layer(layer, quantize_config=_OUTCFG())
            return layer
        ann = keras.models.clone_model(model, clone_function=annot)
        with tfmot.quantization.keras.quantize_scope({"Default8BitOutputQuantizeConfig": _OUTCFG}):
            q = tfmot.quantization.keras.quantize_apply(ann)
        return q, "annotated (output-quantized Multiply/BN)"

def make_unstructured(model, target):
    steps_per_epoch = int(np.ceil(len(Xtr) / BATCH))
    end = steps_per_epoch * max(1, int(0.6 * UNSTR_EPOCHS))
    sched = tfmot.sparsity.keras.PolynomialDecay(0.0, target, 0, end, frequency=max(1, steps_per_epoch//4))
    def wrap(layer):
        if isinstance(layer, (layers.Conv2D, layers.Dense)) and layer.name != "eca_conv":
            return tfmot.sparsity.keras.prune_low_magnitude(layer, pruning_schedule=sched)
        return layer
    return keras.models.clone_model(fresh_copy(model), clone_function=wrap)   # wrapped layers keep the copied weights

In [ ]:
# Cell 9 — TFLite conversion and evaluation
def to_tflite(model, int8=False, calib=None):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    if int8:
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        if calib is not None:
            def rep():
                for i in range(len(calib)):
                    yield [calib[i:i+1].astype(np.float32)]
            conv.representative_dataset = rep
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        conv.inference_input_type = tf.int8
        conv.inference_output_type = tf.int8
    return conv.convert()

def _interp(tfl, threads, batch):
    it = tf.lite.Interpreter(model_content=tfl, num_threads=threads)
    inp = it.get_input_details()[0]
    if batch != 1:
        it.resize_tensor_input(inp["index"], [batch, N, 1])
    it.allocate_tensors()
    return it, it.get_input_details()[0], it.get_output_details()[0]

def _q(x, d):
    if d["dtype"] == np.int8:
        s, z = d["quantization"]
        return np.clip(np.round(x / s + z), -128, 127).astype(np.int8)
    return x.astype(np.float32)

def tflite_predict(tfl, X, batch=256):
    try:
        it, di, do = _interp(tfl, 4, batch)
    except Exception:
        batch = 1; it, di, do = _interp(tfl, 4, 1)
    preds = []
    for i in range(0, len(X), batch):
        xb = X[i:i+batch]; nb = len(xb)
        if nb < batch:
            xb = np.concatenate([xb, np.zeros((batch-nb, N, 1), xb.dtype)])
        it.set_tensor(di["index"], _q(xb, di)); it.invoke()
        preds.append(np.argmax(it.get_tensor(do["index"])[:nb], axis=1))
    return np.concatenate(preds)

def tflite_latency_ms(tfl):
    it, di, _ = _interp(tfl, 1, 1)
    x = _q(TEST["clean"][0][:1], di)
    for _ in range(LAT_WARMUP):
        it.set_tensor(di["index"], x); it.invoke()
    ts = []
    for _ in range(LAT_RUNS):
        t0 = time.perf_counter(); it.set_tensor(di["index"], x); it.invoke(); ts.append(time.perf_counter() - t0)
    return 1000*float(np.median(ts))

In [ ]:
# Cell 10 — static (edge) metrics
def macs_dense(m):
    """Analytic multiply-accumulates of the dense (unpruned-equivalent) graph."""
    tot = 0
    for l in m.layers:
        if isinstance(l, layers.Conv2D):
            _, k, cin, cout = l.kernel.shape
            tot += int(l.output_shape[2]) * int(k) * int(cin) * int(cout)
        elif isinstance(l, layers.Dense):
            tot += int(np.prod(l.kernel.shape))
    return tot

def peak_activation_elems(m):
    """Largest (inputs + output) activation footprint over all layers, in elements."""
    peak = 0
    for l in m.layers:
        if isinstance(l, keras.layers.InputLayer):
            continue
        ins = l.input if isinstance(l.input, list) else [l.input]
        tot = sum(int(np.prod(x.shape[1:])) for x in ins) + int(np.prod(l.output.shape[1:]))
        peak = max(peak, tot)
    return peak

def nonzero_weights(m):
    return int(sum(np.count_nonzero(v.numpy()) for v in m.trainable_weights))

def static_metrics(keras_model, tfl, int8):
    return dict(params=n_trainable(keras_model), nonzero=nonzero_weights(keras_model),
                tflite_kb=len(tfl)/1024, gzip_kb=len(gzip.compress(tfl, 9))/1024,
                macs=macs_dense(keras_model),
                peak_act_kb=peak_activation_elems(keras_model)*(1 if int8 else 4)/1024,
                latency_ms=tflite_latency_ms(tfl))

In [ ]:
# Cell 11 — one complete run (one seed, one architecture)
RES = os.path.join(ROOT, "results")
CSV_ALL = os.path.join(RES, "results_all.csv")
CSV_CLS = os.path.join(RES, "perclass_f1_20dB.csv")
LOG = os.path.join(RES, "run_log.jsonl")

def evaluate_variant(seed, arch, variant, keras_model, tfl, int8, extra):
    st = static_metrics(keras_model, tfl, int8)
    open(os.path.join(ROOT, "models", f"{arch}_{variant}_s{seed}.tflite"), "wb").write(tfl)
    rows, cls_rows, cms = [], [], {}
    for tag, (X, y) in TEST.items():
        p = tflite_predict(tfl, X)
        rows.append(dict(seed=seed, arch=arch, variant=variant, snr=tag,
                         acc=accuracy_score(y, p), macro_f1=f1_score(y, p, average="macro"), **st, **extra))
        if tag == "20dB":
            f1c = f1_score(y, p, average=None, labels=range(NC))
            cls_rows.append(dict(seed=seed, arch=arch, variant=variant, **{CLASSES[i]: f1c[i] for i in range(NC)}))
            cms[f"{arch}|{variant}|{seed}"] = confusion_matrix(y, p, labels=range(NC))
    r20 = [r for r in rows if r["snr"] == "20dB"][0]
    print(f"   {variant:9s} size={st['tflite_kb']:7.1f} KB  F1@clean={rows[0]['macro_f1']:.4f}  "
          f"F1@20dB={r20['macro_f1']:.4f}  F1@10dB={rows[-2]['macro_f1']:.4f}  F1@fdev={rows[-1]['macro_f1']:.4f}  lat={st['latency_ms']:.3f} ms")
    return rows, cls_rows, cms

def run_one(seed, arch):
    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)
    calib_idx = np.random.default_rng(seed).choice(len(Xtr), CALIB_N, replace=False)
    calib = Xtr[calib_idx]                     # noisy + clean mix, drawn from training data only
    rows, cls_rows, cms, info = [], [], {}, {"seed": seed, "arch": arch}
    def add(variant, km, tfl, int8, extra=None):
        r, c, cm = evaluate_variant(seed, arch, variant, km, tfl, int8, extra or {})
        rows.extend(r); cls_rows.extend(c); cms.update(cm)

    # 1) FP32 baseline
    t0 = time.time()
    fp = BUILDERS[arch]()
    h = fit(fp, EPOCHS, LR)
    info["fp32_epochs"] = len(h.history["loss"]); info["fp32_train_s"] = time.time() - t0
    fp.save(os.path.join(ROOT, "models", f"{arch}_fp32_s{seed}.keras"))
    add("fp32", fp, to_tflite(fp), False)

    # 2) PTQ-INT8 (full-integer, calibrated on CALIB_N noisy training signals)
    add("ptq_int8", fp, to_tflite(fp, int8=True, calib=calib), True)

    # 3) QAT-INT8
    q, mode = make_qat(fp); info["qat_mode"] = mode
    fit(q, QAT_EPOCHS, QAT_LR)
    add("qat_int8", fp, to_tflite(q, int8=True, calib=calib), True, {"qat_mode": mode})

    # 4) Iterative structured pruning 25 -> 50 -> 75 %, fine-tuning after every step
    cur = fp
    for frac in PRUNE_STEPS:
        cur, keep = structured_prune(cur, arch, frac)
        fit(cur, FT_EPOCHS, FT_LR)
        tag = f"sp{int(frac*100)}"
        if tag in VARIANTS:
            add(tag, cur, to_tflite(cur), False, {"widths": str(keep)})
            # 5) structured pruning + QAT
            qp, mode = make_qat(cur)
            fit(qp, QAT_EPOCHS, QAT_LR)
            add(tag + "_qat", cur, to_tflite(qp, int8=True, calib=calib), True, {"widths": str(keep), "qat_mode": mode})

    # 6) Unstructured magnitude pruning (Zhu & Gupta 2017 schedule), FP32 storage
    for target in UNSTR_TARGETS:
        pm = make_unstructured(fp, target)
        fit(pm, UNSTR_EPOCHS, FT_LR, extra_cb=[tfmot.sparsity.keras.UpdatePruningStep()],
            patience=UNSTR_EPOCHS, early_stop=False)
        sm = tfmot.sparsity.keras.strip_pruning(pm)
        prunable = [l.kernel.numpy() for l in sm.layers
                    if isinstance(l, (layers.Conv2D, layers.Dense)) and l.name != "eca_conv"]
        achieved = 1 - sum(np.count_nonzero(k) for k in prunable) / sum(k.size for k in prunable)
        assert abs(achieved - target) < 0.02, f"sparsity {achieved:.3f} != target {target}"
        add(f"up{int(target*100)}", sm, to_tflite(sm), False, {"target_sparsity": target})

    info["total_s"] = time.time() - t0
    return rows, cls_rows, cms, info

In [ ]:
# Cell 12 — main loop over seeds and architectures (resumable)
done = set()
if os.path.exists(CSV_ALL) and not QUICK:
    prev = pd.read_csv(CSV_ALL)
    done = set(zip(prev.seed, prev.arch))
else:
    for f in [CSV_ALL, CSV_CLS, LOG]:
        if os.path.exists(f): os.remove(f)

for seed in SEEDS:
    for arch in ARCHS:
        if (seed, arch) in done:
            print(f"skip seed={seed} arch={arch} (already in results)"); continue
        print(f"\n=== seed {seed} | {arch} ===")
        rows, cls_rows, cms, info = run_one(seed, arch)
        pd.DataFrame(rows).to_csv(CSV_ALL, mode="a", header=not os.path.exists(CSV_ALL), index=False)
        pd.DataFrame(cls_rows).to_csv(CSV_CLS, mode="a", header=not os.path.exists(CSV_CLS), index=False)
        np.savez_compressed(os.path.join(RES, f"cm20_{arch}_s{seed}.npz"), **{k.replace("|", "__"): v for k, v in cms.items()})
        with open(LOG, "a") as f:
            f.write(json.dumps(info) + "\n")
        print(f"   done in {info['total_s']/60:.1f} min | QAT mode: {info['qat_mode']}")

In [ ]:
# Cell 13 — summary tables (mean ± std over seeds)
df = pd.read_csv(CSV_ALL)
order = {v: i for i, v in enumerate(VARIANTS)}
snr_order = {snr_tag(s): i for i, s in enumerate(TEST_SNRS)}; snr_order["fdev"] = len(TEST_SNRS)

stat_cols = ["params", "nonzero", "tflite_kb", "gzip_kb", "macs", "peak_act_kb", "latency_ms"]
g = df.groupby(["arch", "variant", "snr"])
summ = g[["acc", "macro_f1"]].agg(["mean", "std"])
summ.columns = ["_".join(c) for c in summ.columns]
summ = summ.join(g[stat_cols].mean()).reset_index()
summ["_v"] = summ.variant.map(order); summ["_s"] = summ.snr.map(snr_order)
summ = summ.sort_values(["arch", "_v", "_s"]).drop(columns=["_v", "_s"])
summ.to_csv(os.path.join(RES, "summary_by_snr.csv"), index=False)

# Main table: one row per (arch, variant) with F1 at each SNR + edge metrics
def pm(m, s): return f"{100*m:.2f} ± {100*(0 if np.isnan(s) else s):.2f}"
rows = []
for (a, v), sub in summ.groupby(["arch", "variant"], sort=False):
    base_kb = summ[(summ.arch == a) & (summ.variant == "fp32")].tflite_kb.iloc[0]
    r = {"Model": a, "Variant": v}
    for s in TEST_SNRS:
        x = sub[sub.snr == snr_tag(s)].iloc[0]
        r[f"F1 {snr_tag(s)} (%)"] = pm(x.macro_f1_mean, x.macro_f1_std)
    x = sub[sub.snr == "fdev"].iloc[0]
    r["F1 freq-dev 30dB (%)"] = pm(x.macro_f1_mean, x.macro_f1_std)
    x = sub.iloc[0]
    r.update({"Params": int(x.params), "Non-zero": int(x.nonzero), "TFLite (KB)": round(x.tflite_kb, 2),
              "gzip (KB)": round(x.gzip_kb, 2), "Compression (x)": round(base_kb / x.tflite_kb, 2),
              "MACs": int(x.macs), "Peak act. (KB)": round(x.peak_act_kb, 2), "Latency (ms)": round(x.latency_ms, 4)})
    rows.append(r)
main = pd.DataFrame(rows)
main.to_csv(os.path.join(RES, "Table_main.csv"), index=False)

cls = pd.read_csv(CSV_CLS).groupby(["arch", "variant"])[CLASSES].mean().reset_index()
cls.to_csv(os.path.join(RES, "Table_perclass_f1_20dB_mean.csv"), index=False)
pd.set_option("display.width", 250, "display.max_columns", 30)
print(main.to_string(index=False))

In [ ]:
# Cell 14 — figures: F1 vs SNR, size–accuracy trade-off, confusion matrices
LABEL = {"fp32": "FP32", "ptq_int8": "PTQ-INT8", "qat_int8": "QAT-INT8", "sp50": "SP-50%", "sp75": "SP-75%",
         "sp50_qat": "SP-50%+QAT", "sp75_qat": "SP-75%+QAT", "up50": "UP-50%", "up75": "UP-75%"}
STYLE = {"fp32": ("k", "o", "-"), "ptq_int8": ("#1f77b4", "s", "--"), "qat_int8": ("#1f77b4", "D", "-"),
         "sp50": ("#d62728", "^", "-"), "sp75": ("#d62728", "v", "--"), "sp50_qat": ("#9467bd", "P", "-"),
         "sp75_qat": ("#9467bd", "X", "--"), "up50": ("#2ca02c", "<", "-"), "up75": ("#2ca02c", ">", "--")}
TITLE = {"cnn": "(a) Plain 1D-CNN", "ccanet": "(b) 1D-CCANet-3"}
xs = list(range(len(TEST_SNRS))); xl = ["Clean" if s is None else str(s) for s in TEST_SNRS]

fig, axs = plt.subplots(1, 2, figsize=(7.0, 3.0), sharey=True)
for ax, a in zip(axs, ARCHS):
    for v in VARIANTS:
        sub = summ[(summ.arch == a) & (summ.variant == v)]
        if sub.empty: continue
        sub = sub.set_index("snr").loc[[snr_tag(s) for s in TEST_SNRS]]
        c, mk, ls = STYLE[v]
        ax.errorbar(xs, 100*sub.macro_f1_mean, yerr=100*sub.macro_f1_std.fillna(0), color=c, marker=mk,
                    ls=ls, ms=3.5, capsize=2, lw=0.9, label=LABEL[v])
    ax.set_xticks(xs); ax.set_xticklabels(xl); ax.set_xlabel("Test SNR (dB)"); ax.set_title(TITLE[a])
    ax.grid(alpha=0.3, lw=0.4)
axs[0].set_ylabel("Macro-F1 (%)")
axs[1].legend(fontsize=6.5, ncol=1, frameon=False, loc="lower left", bbox_to_anchor=(1.0, 0.0))
fig.tight_layout(); fig.savefig(os.path.join(FIG, "Fig_f1_vs_snr.png")); fig.savefig(os.path.join(FIG, "Fig_f1_vs_snr.pdf")); plt.close(fig)

fig, axs = plt.subplots(1, 2, figsize=(7.0, 3.0))
for ax, a in zip(axs, ARCHS):
    for v in VARIANTS:
        x = summ[(summ.arch == a) & (summ.variant == v) & (summ.snr == "20dB")]
        if x.empty: continue
        x = x.iloc[0]; c, mk, _ = STYLE[v]
        size = x.gzip_kb if v.startswith("up") else x.tflite_kb    # sparse models only shrink when compressed
        ax.errorbar(size, 100*x.macro_f1_mean, yerr=100*(0 if np.isnan(x.macro_f1_std) else x.macro_f1_std),
                    color=c, marker=mk, ms=5, capsize=2, ls="none", label=LABEL[v])
    ax.set_xscale("log"); ax.set_xlabel("Model size (KB, log scale)"); ax.set_title(TITLE[a] + ", 20 dB")
    ax.grid(alpha=0.3, lw=0.4, which="both")
axs[0].set_ylabel("Macro-F1 at 20 dB (%)")
axs[1].legend(fontsize=6.5, frameon=False, loc="lower left", bbox_to_anchor=(1.0, 0.0))
fig.tight_layout(); fig.savefig(os.path.join(FIG, "Fig_size_vs_f1_20dB.png")); fig.savefig(os.path.join(FIG, "Fig_size_vs_f1_20dB.pdf")); plt.close(fig)

def summed_cm(arch, variant):
    tot = np.zeros((NC, NC), int)
    for f in glob.glob(os.path.join(RES, f"cm20_{arch}_s*.npz")):
        z = np.load(f)
        for k in z.files:
            if k.startswith(f"{arch}__{variant}__"): tot += z[k]
    return tot

for a in ARCHS:
    fig, axs = plt.subplots(1, 2, figsize=(7.0, 3.6))
    for ax, v in zip(axs, ["fp32", "qat_int8"]):
        cm = summed_cm(a, v); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
        ax.imshow(cmn, cmap="Greys", vmin=0, vmax=1)
        for i in range(NC):
            for j in range(NC):
                if cmn[i, j] >= 0.01:
                    ax.text(j, i, f"{100*cmn[i,j]:.0f}", ha="center", va="center", fontsize=4.5,
                            color="white" if cmn[i, j] > 0.5 else "black")
        ax.set_xticks(range(NC)); ax.set_yticks(range(NC))
        ax.set_xticklabels([f"C{i}" for i in range(NC)], fontsize=5.5, rotation=90)
        ax.set_yticklabels([f"C{i}" for i in range(NC)], fontsize=5.5)
        ax.set_xlabel("Predicted class"); ax.set_ylabel("True class"); ax.set_title(f"{LABEL[v]}", fontsize=8)
    fig.tight_layout(); fig.savefig(os.path.join(FIG, f"Fig_confusion_20dB_{a}.png")); fig.savefig(os.path.join(FIG, f"Fig_confusion_20dB_{a}.pdf")); plt.close(fig)

with open(os.path.join(RES, "environment.json"), "w") as f:
    json.dump({"tensorflow": tf.__version__, "tf_keras": keras.__version__, "tfmot": tfmot.__version__,
               "numpy": np.__version__, "python": platform.python_version(), "cpu": platform.processor(),
               "gpu": [g.name for g in gpus]}, f, indent=2)
print("Figures saved to", FIG)

In [ ]:
# Cell 15 — pack everything for download
import shutil
pack = os.path.join(os.path.dirname(os.path.abspath(ROOT)), "PQD_results_pack")
if os.path.exists(pack): shutil.rmtree(pack)
for sub in ["results", "figures", "models"]:          # the dataset is not packed: it is re-created from the seeds
    shutil.copytree(os.path.join(ROOT, sub), os.path.join(pack, sub))
zip_path = shutil.make_archive(pack, "zip", pack)
print("Download this file from the Output panel:", zip_path)